# 🫁 PULMO·AI™ — Multi-Dataset Mega Training Pipeline (Kaggle GPU)
### Ingesting 9+ CXR Datasets (85,000+ Radiographs) with Zero Patient-Leakage and High-Epoch Dual-Backbone Fine-Tuning

In [ ]:
#!/usr/bin/env python3
"""
=============================================================================
🫁 PULMO·AI™ — Multi-Dataset Mega Training Pipeline (Kaggle Cloud GPU)
=============================================================================
Trains a High-Capacity 3-Class Deep Learning Diagnostic Model
(NORMAL, BACTERIAL, VIRAL / COVID-19) on 9+ Curated Chest Radiograph Datasets
(85,000+ total radiographs) with Strict Zero Patient-Leakage Partitioning,
Dual-Backbone Fine-Tuning (ResNet-50 + DenseNet-121), and Clinical Telemetry.
=============================================================================
"""

import os
import re
import gc
import json
import time
import random
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    balanced_accuracy_score,
    f1_score,
)
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.applications import ResNet50, DenseNet121


In [ ]:
# 1. HARDWARE ACCELERATION & MIXED PRECISION


In [ ]:
print("=" * 75)
print("🫁 PULMO·AI™ — MULTI-DATASET MEGA TRAINING PIPELINE (KAGGLE GPU)")
print("=" * 75)

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print(f"✅ GPU Detected: {len(gpus)} device(s)")
    for gpu in gpus:
        print(f"   • {gpu}")
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception:
            pass
    try:
        from tensorflow.keras import mixed_precision
        mixed_precision.set_global_policy("mixed_float16")
        print("⚡ Mixed Precision (mixed_float16) enabled for maximum Tensor Core throughput.")
    except Exception as e:
        print(f"⚠️ Could not set mixed precision: {e}")
else:
    print("ℹ️ No GPU detected. Running in standard CPU mode.")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


In [ ]:
# 2. MULTI-DATASET SCANNING & ZERO-LEAKAGE EXTRACTION


In [ ]:
def extract_patient_id(filename: str) -> str:
    """Extract patient identifier from various hospital naming conventions."""
    fn = filename.lower()
    m_person = re.match(r"^(person\d+)_", fn)
    if m_person:
        return m_person.group(1)
    m_norm = re.match(r"^(normal2-im-\d+|im-\d+)", fn)
    if m_norm:
        return m_norm.group(1)
    m_paren = re.match(r"^([a-z0-9_-]+)\((\d+)\)", fn)
    if m_paren:
        return f"{m_paren.group(1)}_{m_paren.group(2)}"
    m_covid_num = re.match(r"^(covid[_-]?\d+)", fn)
    if m_covid_num:
        return m_covid_num.group(1)
    return fn.split(".")[0]


def classify_path(p_str: str) -> str:
    """
    Classify radiograph into:
      - 'NORMAL'
      - 'VIRAL' (Viral Pneumonia, SARS-CoV-2 / COVID-19)
      - 'BACTERIAL' (Bacterial Pneumonia, Lobar Consolidation)
    Or None if mask, segmentation, or non-diagnostic file.
    """
    p_lower = p_str.lower()
    
    # 1. Skip segmentation masks, heatmaps, and annotation overlays
    if "mask" in p_lower or ("segmentation" in p_lower and "image" not in p_lower):
        return None
    
    parts = p_str.split("/")
    fn = parts[-1].lower()
    parent = parts[-2].lower() if len(parts) >= 2 else ""
    grandparent = parts[-3].lower() if len(parts) >= 3 else ""

    # 2. Check viral / covid
    if (
        "virus" in fn or "viral" in fn
        or "virus" in parent or "viral" in parent
        or "viral pneumonia" in grandparent
        or "covid" in fn or "covid" in parent or "covid" in grandparent
    ):
        return "VIRAL"

    # 3. Check normal
    if (
        "normal" in fn or "normal" in parent or "normal" in grandparent
        or re.match(r"^(normal2-im-|im-\d+)", fn)
    ):
        return "NORMAL"

    # 4. Check bacterial
    if (
        "bacteria" in fn or "bacteria" in parent or "bacteria" in grandparent
        or "streptococcus" in fn or "streptococcus" in parent
    ):
        return "BACTERIAL"

    # 5. Generic pneumonia directory (predominantly bacterial infiltrate)
    if "pneumonia" in fn or "pneumonia" in parent or "pneumonia" in grandparent:
        return "BACTERIAL"

    return None


def scan_all_datasets() -> pd.DataFrame:
    """Scan all mounted datasets in /kaggle/input or local data/ folders."""
    kaggle_input = Path("/kaggle/input")
    search_dirs = []

    if kaggle_input.exists():
        subdirs = [p for p in kaggle_input.iterdir() if p.is_dir()]
        print(f"📂 Mounted Kaggle datasets ({len(subdirs)}): {[p.name for p in subdirs]}")
        search_dirs.extend(subdirs)
    else:
        local_candidates = [Path("data"), Path("../data")]
        search_dirs.extend([c for c in local_candidates if c.exists()])

    if not search_dirs:
        raise FileNotFoundError("No datasets found in /kaggle/input or local data/.")

    print("\n🔍 Scanning and indexing all radiographs across all mounted datasets...")
    all_records = []
    seen_hashes = set()

    for d in search_dirs:
        count_before = len(all_records)
        for p in d.glob("**/*.*"):
            # Skip hidden files and macOS metadata
            if p.name.startswith(".") or p.name.startswith("._") or "__macosx" in str(p).lower():
                continue

            # Check valid image extension
            if not (p.is_file() and p.suffix.lower() in (".jpeg", ".jpg", ".png")):
                continue

            # Check file size (> 2KB to filter corrupted / empty files)
            try:
                size = p.stat().st_size
                if size < 2048:
                    continue
            except Exception:
                continue

            # Classify label
            label = classify_path(str(p))
            if label is None:
                continue

            # Deduplication key: filename + file size
            dedup_key = f"{p.name.lower()}_{size}"
            if dedup_key in seen_hashes:
                continue
            seen_hashes.add(dedup_key)

            pid = extract_patient_id(p.name)
            all_records.append({
                "filepath": str(p),
                "filename": p.name,
                "dataset": d.name,
                "patient_id": pid,
                "label": label,
            })
        added = len(all_records) - count_before
        print(f"   • Dataset '{d.name}': +{added:,} radiographs indexed.")

    df = pd.DataFrame(all_records)
    return df


df_all = scan_all_datasets()
print(f"\n📊 Total unique radiographs indexed: {len(df_all):,}")
print(f"👥 Unique patients found:            {df_all['patient_id'].nunique():,}")
print("📈 Class breakdown across mega dataset:")
print(df_all["label"].value_counts().to_string())


In [ ]:
# 3. HERMETIC PATIENT-LEVEL ZERO-LEAKAGE SPLIT


In [ ]:
patient_label_map = df_all.groupby("patient_id")["label"].first()
patients_by_class = defaultdict(list)
for pid, lbl in patient_label_map.items():
    patients_by_class[lbl].append(pid)

train_patients, val_patients, test_patients = [], [], []
SPLIT_RATIOS = (0.75, 0.125, 0.125)  # 75% Train, 12.5% Val, 12.5% Test

for lbl, pids in patients_by_class.items():
    random.shuffle(pids)
    n = len(pids)
    n_train = int(n * SPLIT_RATIOS[0])
    n_val = int(n * SPLIT_RATIOS[1])

    train_patients.extend(pids[:n_train])
    val_patients.extend(pids[n_train:n_train + n_val])
    test_patients.extend(pids[n_train + n_val:])

train_set = set(train_patients)
val_set = set(val_patients)
test_set = set(test_patients)

# Audit patient leakage
overlap_train_val = train_set.intersection(val_set)
overlap_train_test = train_set.intersection(test_set)
overlap_val_test = val_set.intersection(test_set)
assert len(overlap_train_val) == 0 and len(overlap_train_test) == 0 and len(overlap_val_test) == 0
print("\n🔒 ZERO-LEAKAGE AUDIT PASSED: 0.0% patient overlap across all splits!")

df_train = df_all[df_all["patient_id"].isin(train_set)].copy().reset_index(drop=True)
df_val = df_all[df_all["patient_id"].isin(val_set)].copy().reset_index(drop=True)
df_test = df_all[df_all["patient_id"].isin(test_set)].copy().reset_index(drop=True)

print(f"Train images: {len(df_train):,} (Patients: {len(train_set):,})")
print(f"Val images:   {len(df_val):,} (Patients: {len(val_set):,})")
print(f"Test images:  {len(df_test):,} (Patients: {len(test_set):,})")

CLASS_NAMES = ["NORMAL", "BACTERIAL", "VIRAL"]
LABEL_MAP = {c: i for i, c in enumerate(CLASS_NAMES)}

df_train["target"] = df_train["label"].map(LABEL_MAP)
df_val["target"] = df_val["label"].map(LABEL_MAP)
df_test["target"] = df_test["label"].map(LABEL_MAP)

# Calculate balanced class weights
y_train_raw = df_train["target"].values
classes = np.unique(y_train_raw)
weights = compute_class_weight("balanced", classes=classes, y=y_train_raw)
class_weights_dict = {int(c): float(w) for c, w in zip(classes, weights)}
print(f"⚖️ Balanced Class Weights: {class_weights_dict}")


In [ ]:
# 4. TF.DATA HIGH-PERFORMANCE STREAMING PIPELINE


In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 64 if gpus else 16
AUTOTUNE = tf.data.AUTOTUNE


def load_and_preprocess_image(path, label):
    img_raw = tf.io.read_file(path)
    # Decode jpeg/png safely
    img = tf.image.decode_image(img_raw, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    return img, label


def augment_image(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, max_delta=0.08)
    img = tf.image.random_contrast(img, lower=0.92, upper=1.08)
    return img, label


train_ds = (
    tf.data.Dataset.from_tensor_slices((df_train["filepath"].values, df_train["target"].values))
    .shuffle(buffer_size=min(len(df_train), 10000), seed=SEED)
    .map(load_and_preprocess_image, num_parallel_calls=AUTOTUNE)
    .map(augment_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((df_val["filepath"].values, df_val["target"].values))
    .map(load_and_preprocess_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

test_ds = (
    tf.data.Dataset.from_tensor_slices((df_test["filepath"].values, df_test["target"].values))
    .map(load_and_preprocess_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)


In [ ]:
# 5. DEEP BACKBONE ARCHITECTURE (ResNet-50 & DenseNet-121)


In [ ]:
def build_chexnet_3class(backbone_type: str = "densenet121") -> tuple:
    """Build CheXNet 3-class model with transfer learning backbone."""
    inputs = keras.Input(shape=(224, 224, 3), name="cxr_input")

    if backbone_type.lower() == "resnet50":
        base = ResNet50(weights="imagenet", include_top=False, input_tensor=inputs)
    elif backbone_type.lower() == "densenet121":
        base = DenseNet121(weights="imagenet", include_top=False, input_tensor=inputs)
    else:
        raise ValueError(f"Unsupported backbone: {backbone_type}")

    base.trainable = False

    x = layers.GlobalAveragePooling2D(name="gap")(base.output)
    x = layers.Dense(256, activation="relu", kernel_regularizer=regularizers.l2(1e-4), name="dense_feat")(x)
    x = layers.BatchNormalization(name="feat_bn")(x)
    x = layers.Dropout(0.4, name="feat_dropout")(x)
    outputs = layers.Dense(3, activation="softmax", dtype="float32", name="class_output")(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name=f"chexnet_{backbone_type}_3class")
    return model, base


In [ ]:
# 6. TWO-STAGE HIGH-EPOCH TRAINING SCHEDULE


In [ ]:
def train_model_two_stages(
    model: keras.Model,
    base_model: keras.Model,
    name: str,
    stage1_epochs: int = 5,
    stage2_epochs: int = 20,
    output_dir: Path = Path("/kaggle/working/models"),
) -> dict:
    output_dir.mkdir(parents=True, exist_ok=True)
    best_weight_path = output_dir / f"{name}_best.h5"

    print(f"\n=======================================================")
    print(f"  TRAINING {name.upper()}: STAGE 1 (FROZEN HEAD WARMUP, {stage1_epochs} EPOCHS)")
    print(f"=======================================================")

    opt_stage1 = keras.optimizers.Adam(learning_rate=1e-3)
    model.compile(
        optimizer=opt_stage1,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    cb_stage1 = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5),
    ]

    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=stage1_epochs,
        class_weight=class_weights_dict,
        callbacks=cb_stage1,
        verbose=1,
    )

    print(f"\n=======================================================")
    print(f"  TRAINING {name.upper()}: STAGE 2 (DEEP END-TO-END CONV TUNING, {stage2_epochs} EPOCHS)")
    print(f"=======================================================")

    base_model.trainable = True
    if "resnet" in name.lower():
        for layer in base_model.layers:
            layer.trainable = True if "conv5" in layer.name else False
    elif "densenet" in name.lower():
        for layer in base_model.layers:
            layer.trainable = True if ("conv5" in layer.name or "block4" in layer.name) else False

    trainable_count = sum(len(layer.weights) for layer in model.layers if layer.trainable)
    print(f"🔓 Fine-tuning enabled. Trainable weight tensors: {trainable_count}")

    opt_stage2 = keras.optimizers.Adam(learning_rate=1e-5)
    model.compile(
        optimizer=opt_stage2,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    cb_stage2 = [
        keras.callbacks.ModelCheckpoint(
            filepath=str(best_weight_path),
            monitor="val_loss",
            save_best_only=True,
            verbose=1,
        ),
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7),
    ]

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=stage2_epochs,
        class_weight=class_weights_dict,
        callbacks=cb_stage2,
        verbose=1,
    )

    return history


In [ ]:
# 7. EXECUTION


In [ ]:
output_path = Path("/kaggle/working/models") if Path("/kaggle/working").exists() else Path("models/kaggle_export")
output_path.mkdir(parents=True, exist_ok=True)

# Train DenseNet-121 (Primary SOTA Medical Architecture)
print("\n>>> [1/2] Training DenseNet-121 on Mega Dataset...")
densenet_model, densenet_base = build_chexnet_3class("densenet121")
train_model_two_stages(densenet_model, densenet_base, "densenet121_mega", stage1_epochs=5, stage2_epochs=20, output_dir=output_path)

# Save as primary subtype model
densenet_model.save(str(output_path / "subtype_model_mega.h5"))
print(f"💾 Saved primary subtype model to {output_path / 'subtype_model_mega.h5'}")

# Train ResNet-50 (Ensemble Partner)
print("\n>>> [2/2] Training ResNet-50 on Mega Dataset...")
resnet_model, resnet_base = build_chexnet_3class("resnet50")
train_model_two_stages(resnet_model, resnet_base, "resnet50_mega", stage1_epochs=5, stage2_epochs=15, output_dir=output_path)


In [ ]:
# 8. ENSEMBLE EVALUATION ON HELD-OUT PATIENTS


In [ ]:
print("\n" + "=" * 75)
print("  🏆 HELD-OUT PATIENT ENSEMBLE TEST EVALUATION (MEGA COHORT)")
print("=" * 75)

print("Running batch inference across held-out test cohort...")
probs_dense = densenet_model.predict(test_ds, verbose=1)
probs_resnet = resnet_model.predict(test_ds, verbose=1)

probs_ensemble = 0.55 * probs_dense + 0.45 * probs_resnet
y_true = df_test["target"].values
y_pred = np.argmax(probs_ensemble, axis=1)

bal_acc = balanced_accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")

print(f"\n✨ Mega-Cohort Balanced Accuracy: {bal_acc * 100:.2f}%")
print(f"✨ Mega-Cohort Macro-F1 Score:    {macro_f1:.4f}")
print("\nClassification Report (Held-out Patients):")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:")
cm_df = pd.DataFrame(cm, index=[f"True {c}" for c in CLASS_NAMES], columns=[f"Pred {c}" for c in CLASS_NAMES])
print(cm_df)

# Telemetry export
meta = {
    "date": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "total_images_indexed": len(df_all),
    "unique_patients_indexed": int(df_all["patient_id"].nunique()),
    "train_images": len(df_train),
    "val_images": len(df_val),
    "test_images": len(df_test),
    "classes": CLASS_NAMES,
    "metrics": {
        "balanced_accuracy": float(bal_acc),
        "macro_f1": float(macro_f1),
    },
    "confusion_matrix": cm.tolist(),
}
with open(output_path / "mega_dataset_telemetry.json", "w") as f:
    json.dump(meta, f, indent=2)

print(f"\n✅ Training pipeline completed successfully! Outputs saved to {output_path.resolve()}")
print("🚀 Ready to download to local models/current/ for deployment!")
